In [26]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Define named lambda functions globally for proper serialization/deserialization
def squeeze_last_axis(z):
    return tf.squeeze(z, axis=-1)

def expand_for_multiply(z):
    return tf.expand_dims(
        tf.expand_dims(
            tf.expand_dims(z, axis=-1),
            axis=-1
        ),
        axis=-1
    )

def reduce_sum_time(z):
    return tf.reduce_sum(z, axis=1)

# ============================================================
# TEMPORAL ATTENTION U-NET
# U-TAE-STYLE ARCHITECTURE
# ============================================================

def build_temporal_utae(
    time_steps=15,
    height=128,
    width=128,
    channels=5
):

    inputs = layers.Input(
        shape=(time_steps, height, width, channels),
        name="temporal_input"
    )

    # --------------------------------------------------------
    # Shared spatial encoder
    # The SAME CNN is applied to every time step.
    # --------------------------------------------------------

    x = layers.TimeDistributed(
        layers.Conv2D(
            16, 3,
            padding="same",
            activation="relu"
        )
    )(inputs)

    x = layers.TimeDistributed(
        layers.Conv2D(
            16, 3,
            padding="same",
            activation="relu"
        )
    )(x)

    # Save high-resolution temporal features
    skip1 = x

    x = layers.TimeDistributed(
        layers.MaxPooling2D(pool_size=2)
    )(x)

    # --------------------------------------------------------
    # Encoder level 2
    # --------------------------------------------------------

    x = layers.TimeDistributed(
        layers.Conv2D(
            32, 3,
            padding="same",
            activation="relu"
        )
    )(x)

    x = layers.TimeDistributed(
        layers.Conv2D(
            32, 3,
            padding="same",
            activation="relu"
        )
    )(x)

    skip2 = x

    x = layers.TimeDistributed(
        layers.MaxPooling2D(pool_size=2)
    )(x)

    # --------------------------------------------------------
    # Encoder level 3
    # --------------------------------------------------------

    x = layers.TimeDistributed(
        layers.Conv2D(
            64, 3,
            padding="same",
            activation="relu"
        )
    )(x)

    x = layers.TimeDistributed(
        layers.Conv2D(
            64, 3,
            padding="same",
            activation="relu"
        )
    )(x)

    # --------------------------------------------------------
    # TEMPORAL ATTENTION
    #
    # x shape:
    # (batch, time, 32, 32, 64)
    #
    # Convert spatial features into a temporal descriptor.
    # --------------------------------------------------------

    temporal_descriptor = layers.TimeDistributed(
        layers.GlobalAveragePooling2D()
    )(x)

    # temporal_descriptor:
    # (batch, time, 64)

    # Learn attention score for each time step
    attention_scores = layers.Dense(
        1,
        activation=None,
        name="temporal_attention_scores"
    )(temporal_descriptor)

    # Remove last dimension
    attention_scores = layers.Lambda(
        squeeze_last_axis,
        output_shape=(time_steps,)
    )(attention_scores)

    # Normalize across the 15 time steps
    attention_weights = layers.Softmax(
        axis=1,
        name="temporal_attention_weights"
    )(attention_scores)

    # Restore dimension for multiplication
    attention_weights_expanded = layers.Lambda(
        expand_for_multiply,
        output_shape=(time_steps, 1, 1, 1)
    )(attention_weights)

    # Weighted temporal aggregation
    weighted_x = layers.Multiply()([x, attention_weights_expanded])

    x = layers.Lambda(
        reduce_sum_time,
        output_shape=(height // 4, width // 4, 64) # 128/4 = 32
    )(weighted_x)

    # --------------------------------------------------------
    # TEMPORAL SKIP CONNECTIONS
    #
    # Apply same attention weights to skip features.
    # --------------------------------------------------------

    # Lambda for expanding weights for skip connections
    expand_weights_lambda = layers.Lambda(
        expand_for_multiply,
        output_shape=(time_steps, 1, 1, 1)
    )

    skip2_weighted = layers.Multiply()([
        skip2,
        expand_weights_lambda(attention_weights)
    ])

    skip2_fused = layers.Lambda(
        reduce_sum_time,
        output_shape=(height // 2, width // 2, 32) # 128/2 = 64
    )(skip2_weighted)

    skip1_weighted = layers.Multiply()([
        skip1,
        expand_weights_lambda(attention_weights)
    ])

    skip1_fused = layers.Lambda(
        reduce_sum_time,
        output_shape=(height, width, 16)
    )(skip1_weighted)

    # --------------------------------------------------------
    # DECODER
    # --------------------------------------------------------

    x = layers.UpSampling2D()(x)

    x = layers.Concatenate()([
        x,
        skip2_fused
    ])

    x = layers.Conv2D(
        32, 3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.Conv2D(
        32, 3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.UpSampling2D()(x)

    x = layers.Concatenate()([
        x,
        skip1_fused
    ])

    x = layers.Conv2D(
        16, 3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.Conv2D(
        16, 3,
        padding="same",
        activation="relu"
    )(x)

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid",
        name="segmentation_output"
    )(x)

    return models.Model(
        inputs,
        outputs,
        name="Temporal_U_TAE"
    )

In [27]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

print("\nTesting basic Conv2D...")

test_input = tf.keras.Input(
    shape=(15, 128, 128, 5)
)

test_output = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Conv2D(
        16, 3, padding="same", activation="relu"
    )
)(test_input)

test_model = tf.keras.Model(test_input, test_output)

test_model.summary()

print("\nMODEL CREATED SUCCESSFULLY")

TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Testing basic Conv2D...


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 15, 128, 128,   │             0 │
│                                 │ 5)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 15, 128, 128,   │           736 │
│ (TimeDistributed)               │ 16)                    │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 736 (2.88 KB)

 Trainable params: 736 (2.88 KB)

 Non-trainable params: 0 (0.00 B)


MODEL CREATED SUCCESSFULLY


In [28]:
utae_model = build_temporal_utae(
    time_steps=15,
    height=128,
    width=128,
    channels=5
)

print("U-TAE model created successfully!")
utae_model.summary()

U-TAE model created successfully!


Model: "Temporal_U_TAE"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ temporal_input      │ (None, 15, 128,   │          0 │ -                 │
│ (InputLayer)        │ 128, 5)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 15, 128,   │        736 │ temporal_input[0… │
│ (TimeDistributed)   │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 15, 128,   │      2,320 │ time_distributed… │
│ (TimeDistributed)   │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 15, 64,    │          0 │ time_distributed… │
│ (TimeDistributed)   │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_4  │ (None, 15, 64,    │      4,640 │ time_distributed… │
│ (TimeDistributed)   │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_5  │ (None, 15, 64,    │      9,248 │ time_distributed… │
│ (TimeDistributed)   │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_6  │ (None, 15, 32,    │          0 │ time_distributed… │
│ (TimeDistributed)   │ 32, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_7  │ (None, 15, 32,    │     18,496 │ time_distributed… │
│ (TimeDistributed)   │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_8  │ (None, 15, 32,    │     36,928 │ time_distributed… │
│ (TimeDistributed)   │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_9  │ (None, 15, 64)    │          0 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attention… │ (None, 15, 1)     │         65 │ time_distributed… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 15)        │          0 │ temporal_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attention… │ (None, 15)        │          0 │ lambda[0][0]      │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 15, 1, 1,  │          0 │ temporal_attenti… │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 15, 32,    │          0 │ time_distributed… │
│                     │ 32, 64)           │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 15, 1, 1,  │          0 │ temporal_attenti… │
│                     │ 1)                │            │ temporal_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 32, 32,    │          0 │ multiply[0][0]    │
│                     │ 64)               │            │                 

 Total params: 118,626 (463.38 KB)

 Trainable params: 118,626 (463.38 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
import numpy as np

# Load using memory mapping
X = np.load(
    "/content/data/temporal_fused/X_temporal_fused.npy",
    mmap_mode="r"
)

y = np.load(
    "/content/data/temporal_fused/y_temporal_fused.npy",
    mmap_mode="r"
)

print("X:", X.shape)
print("y:", y.shape)

# --------------------------------------------------
# Reproducible 80/20 split
# --------------------------------------------------

np.random.seed(42)

indices = np.random.permutation(len(X))

train_idx = indices[:400]
test_idx  = indices[400:]

print("\nTraining samples:", len(train_idx))
print("Testing samples :", len(test_idx))

print("\nTrain indices:", train_idx.shape)
print("Test indices :", test_idx.shape)

X: (500, 15, 128, 128, 5)
y: (500, 128, 128, 1)

Training samples: 400
Testing samples : 100

Train indices: (400,)
Test indices : (100,)


In [30]:
import tensorflow as tf
import numpy as np

BATCH_SIZE = 4

class TemporalDataGenerator(tf.keras.utils.Sequence):

    def __init__(self, X_path, y_path, indices, batch_size=4, shuffle=True):
        self.X_path = X_path
        self.y_path = y_path
        self.indices = np.array(indices)
        self.batch_size = batch_size
        self.shuffle = shuffle

        self.X = np.load(self.X_path, mmap_mode="r")
        self.y = np.load(self.y_path, mmap_mode="r")

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):

        batch_indices = self.indices[
            index * self.batch_size:
            (index + 1) * self.batch_size
        ]

        # Load ONLY this batch
        X_batch = np.asarray(
            self.X[batch_indices],
            dtype=np.float32
        )

        y_batch = np.asarray(
            self.y[batch_indices],
            dtype=np.float32
        )

        return X_batch, y_batch

    def on_epoch_end(self):

        if self.shuffle:
            np.random.shuffle(self.indices)


# --------------------------------------------------
# Create generators
# --------------------------------------------------

train_gen = TemporalDataGenerator(
    "/content/data/temporal_fused/X_temporal_fused.npy",
    "/content/data/temporal_fused/y_temporal_fused.npy",
    train_idx,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_gen = TemporalDataGenerator(
    "/content/data/temporal_fused/X_temporal_fused.npy",
    "/content/data/temporal_fused/y_temporal_fused.npy",
    test_idx,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_gen))
print("Test batches :", len(test_gen))

# Test ONE batch
X_batch, y_batch = train_gen[0]

print("\nOne batch:")
print("X batch:", X_batch.shape)
print("y batch:", y_batch.shape)
print("X dtype:", X_batch.dtype)
print("y dtype:", y_batch.dtype)

Train batches: 100
Test batches : 25

One batch:
X batch: (4, 15, 128, 128, 5)
y batch: (4, 128, 128, 1)
X dtype: float32
y dtype: float32


In [31]:
import tensorflow as tf

# ============================================================
# DICE COEFFICIENT
# ============================================================

def dice_coef(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (
        (2.0 * intersection + smooth) /
        (
            tf.reduce_sum(y_true) +
            tf.reduce_sum(y_pred) +
            smooth
        )
    )


# ============================================================
# BCE + DICE LOSS
# ============================================================

def bce_dice_loss(y_true, y_pred):

    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred
    )

    bce = tf.reduce_mean(bce)

    dice_loss = 1.0 - dice_coef(
        y_true,
        y_pred
    )

    return bce + dice_loss


# ============================================================
# COMPILE U-TAE
# ============================================================

utae_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=bce_dice_loss,
    metrics=[
        dice_coef
    ]
)

print("U-TAE compiled successfully!")

U-TAE compiled successfully!


In [32]:
# ============================================================
# U-TAE TRAINING
# ============================================================

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "/content/data/temporal_fused/best_utae.keras",
        monitor="val_dice_coef",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coef",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

history = utae_model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - dice_coef: 0.0222 - loss: 1.3762
Epoch 1: val_dice_coef improved from None to 0.01054, saving model to /content/data/temporal_fused/best_utae.keras

Epoch 1: finished saving model to /content/data/temporal_fused/best_utae.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 36s 96ms/step - dice_coef: 0.0181 - loss: 1.2017 - val_dice_coef: 0.0105 - val_loss: 1.0677 - learning_rate: 1.0000e-04
Epoch 2/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - dice_coef: 0.0157 - loss: 1.0756
Epoch 2: val_dice_coef did not improve from 0.01054
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 60ms/step - dice_coef: 0.0142 - loss: 1.0745 - val_dice_coef: 0.0105 - val_loss: 1.0639 - learning_rate: 1.0000e-04
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - dice_coef: 0.0127 - loss: 1.0677
Epoch 3: val_dice_coef improved from 0.01054 to 0.01139, saving model to /content/data/temporal_fused/best_utae.keras

Epoch 3: finished saving model to /content/data/temporal_fused/best_utae.keras
100/1

In [33]:
import tensorflow as tf
import numpy as np

# Load the best saved U-TAE model
best_utae = tf.keras.models.load_model(
    "/content/data/temporal_fused/best_utae.keras",
    custom_objects={
        "bce_dice_loss": bce_dice_loss,
        "dice_coef": dice_coef,
        "squeeze_last_axis": squeeze_last_axis,
        "expand_for_multiply": expand_for_multiply,
        "reduce_sum_time": reduce_sum_time
    },
    safe_mode=False # Add this argument to allow deserialization of Lambda layers
)

print("Best U-TAE model loaded successfully!")

# Evaluate on the held-out test set
results = best_utae.evaluate(
    test_gen,
    verbose=1
)

print("\n========== FINAL U-TAE TEST RESULT ==========")
print("Test Loss :", results[0])
print("Test Dice :", results[1])

Best U-TAE model loaded successfully!
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - dice_coef: 0.3543 - loss: 0.6975

========== FINAL U-TAE TEST RESULT ==========
Test Loss : 0.6974906921386719
Test Dice : 0.3542763888835907


In [34]:
import numpy as np
import tensorflow as tf

# Load best model
best_utae = tf.keras.models.load_model(
    "/content/data/temporal_fused/best_utae.keras",
    custom_objects={
        "bce_dice_loss": bce_dice_loss,
        "dice_coef": dice_coef,
        "squeeze_last_axis": squeeze_last_axis,
        "expand_for_multiply": expand_for_multiply,
        "reduce_sum_time": reduce_sum_time
    },
    safe_mode=False
)

# Get predictions for one test batch
X_test_batch, y_test_batch = test_gen[0]

pred = best_utae.predict(
    X_test_batch,
    verbose=0
)

print("========== PREDICTION CHECK ==========")

print("Prediction shape:", pred.shape)

print("Prediction min:", pred.min())
print("Prediction max:", pred.max())
print("Prediction mean:", pred.mean())

print("\nGround truth mean:", y_test_batch.mean())

# Try threshold 0.5
pred_binary = (pred >= 0.5).astype(np.float32)

print("\nPredicted positive pixel %:",
      100 * pred_binary.mean())

print("Actual positive pixel %:",
      100 * y_test_batch.mean())

========== PREDICTION CHECK ==========
Prediction shape: (4, 128, 128, 1)
Prediction min: 5.857505e-10
Prediction max: 0.9973085
Prediction mean: 0.0059801107

Ground truth mean: 0.005004883

Predicted positive pixel %: 0.55389404
Actual positive pixel %: 0.5004883
